In [ ]:
import pandas as pd, numpy as np, os
import matplotlib.pyplot as plt, matplotlib.ticker as mticker

FINAL_PATH   = r"C:\Drone_Attack_Similarity_Project\DATASET\Final"
TABLES_PATH  = r"C:\Drone_Attack_Similarity_Project\Rapport\tables"
FIGURES_PATH = r"C:\Drone_Attack_Similarity_Project\Rapport\figures"
os.makedirs(TABLES_PATH, exist_ok=True)
os.makedirs(FIGURES_PATH, exist_ok=True)

datasets_final = {}
for fname in os.listdir(FINAL_PATH):
    if fname.endswith("_final.csv"):
        name = fname.replace("_final.csv", "").replace("_", "-")
        datasets_final[name] = pd.read_csv(FINAL_PATH + "\\" + fname)

print(f"{len(datasets_final)} datasets finaux chargés.")

MCP_UAVIDS = {
    "M": [
        "TxPackets", "RxPackets", "LostPackets", "TxBytes", "RxBytes",
        "TxPacketRate/s", "RxPacketRate/s", "TxByteRate/s", "RxByteRate/s",
        "PacketDropRate",
    ],
    "C": [
        "Protocol", "SrcPort", "DstPort",
    ],
    "P": [
        "FlowDuration/s", "MeanDelay/s", "MeanJitter/s",
        "Throughput/Kbps", "MeanPacketSize", "AverageHopCount",
    ],
}

MCP_COMPOSITE = {
    "M": [
        "Total Fwd Packets", "Total Backward Packets",
        "Total Length of Fwd Packets", "Total Length of Bwd Packets",
        "Fwd Packet Length Max", "Fwd Packet Length Min", "Fwd Packet Length Mean",
        "Bwd Packet Length Max", "Bwd Packet Length Min", "Bwd Packet Length Mean",
        "Flow Packets/s", "Fwd Packets/s", "Bwd Packets/s",
    ],
    "C": [
        "Protocol", "Source Port", "Destination Port",
        "Fwd Header Length", "Bwd Header Length",
    ],
    "P": [
        "Flow Duration", "Flow IAT Mean", "Flow IAT Std",
        "Flow IAT Max", "Flow IAT Min", "Fwd IAT Total", "Fwd IAT Mean",
        "Bwd IAT Total", "Bwd IAT Mean", "Flow Bytes/s",
        "Average Packet Size", "Avg Fwd Segment Size", "Avg Bwd Segment Size",
    ],
}

print("Mapping UAVIDS-2025 :")
for dim, feats in MCP_UAVIDS.items():
    print(f"  {dim} ({len(feats)} features) : {feats}")
print()
print("Mapping Composite IoT/UAV :")
for dim, feats in MCP_COMPOSITE.items():
    print(f"  {dim} ({len(feats)} features) : {feats}")



df_u = datasets_final.get("UAVIDS-2025")
if df_u is not None:
    print("Vérification UAVIDS-2025 :")
    for dim, feats in MCP_UAVIDS.items():
        ok      = [f for f in feats if f in df_u.columns]
        missing = [f for f in feats if f not in df_u.columns]
        status  = "" if not missing else f"manque : {missing}"
        print(f"  Dim {dim} : {len(ok)}/{len(feats)} présentes  {status}")

df_comp = datasets_final.get("Bruteforce")
if df_comp is not None:
    print("\nVérification Composite (Bruteforce) :")
    for dim, feats in MCP_COMPOSITE.items():
        ok      = [f for f in feats if f in df_comp.columns]
        missing = [f for f in feats if f not in df_comp.columns]
        status  = "" if not missing else f" manque : {missing}"
        print(f"  Dim {dim} : {len(ok)}/{len(feats)} présentes  {status}")

def mapping_to_df(mapping, dataset_name):
    return pd.DataFrame([
        {"Feature": f, "Dimension": dim, "Dataset": dataset_name}
        for dim, feats in mapping.items()
        for f in feats
    ])

df_map_uavids    = mapping_to_df(MCP_UAVIDS,    "UAVIDS-2025")
df_map_composite = mapping_to_df(MCP_COMPOSITE, "Composite IoT/UAV")
df_map_all       = pd.concat([df_map_uavids, df_map_composite], ignore_index=True)

df_map_uavids.to_csv(   TABLES_PATH + "\\mapping_MCP_UAVIDS.csv",    index=False)
df_map_composite.to_csv(TABLES_PATH + "\\mapping_MCP_Composite.csv", index=False)
df_map_all.to_csv(      TABLES_PATH + "\\mapping_MCP_complet.csv",   index=False)

print(f" mapping_MCP_UAVIDS.csv     ({len(df_map_uavids)} lignes)")
print(f" mapping_MCP_Composite.csv  ({len(df_map_composite)} lignes)")
print(f" mapping_MCP_complet.csv    ({len(df_map_all)} lignes)")
df_map_all.groupby(["Dataset", "Dimension"]).size().reset_index(name="nb_features")


COLOR_M, COLOR_C, COLOR_P = "#2196F3", "#4CAF50", "#FF9800"

pivot     = df_map_all.groupby(["Dataset", "Dimension"]).size().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot.plot(kind="bar", ax=axes[0], stacked=True,
           color=[COLOR_M, COLOR_C, COLOR_P], edgecolor="white")
axes[0].set_title("Nb features par dimension M/C/P", fontweight="bold")
axes[0].set_xlabel("Dataset")
axes[0].set_ylabel("Nb features")
axes[0].tick_params(axis="x", rotation=10)
axes[0].legend(title="Dimension")

pivot_pct.plot(kind="bar", ax=axes[1], stacked=True, color=[COLOR_M, COLOR_C, COLOR_P], edgecolor="white")
axes[1].set_title("Répartition M/C/P (%) par dataset", fontweight="bold")
axes[1].set_xlabel("Dataset")
axes[1].set_ylabel("%")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[1].tick_params(axis="x", rotation=10)
axes[1].legend(title="Dimension")

plt.tight_layout()
plt.savefig(FIGURES_PATH + "\\mapping_MCP_distribution.png", dpi=150)
plt.show()
print("mapping_MCP_distribution.png")



Datasets finaux chargés :
  Bruteforce         → 5097 lignes, 45 colonnes
  DDoS               → 14292 lignes, 45 colonnes
  DoS                → 14121 lignes, 45 colonnes
  Evil               → 10 lignes, 45 colonnes
  Fakelanding        → 69 lignes, 45 colonnes
  MITM               → 208 lignes, 45 colonnes
  Normal             → 36806 lignes, 45 colonnes
  Reconnassiance     → 1152 lignes, 45 colonnes
  Reply              → 202 lignes, 45 colonnes
  Scanning           → 1152 lignes, 45 colonnes
  UAVIDS-2025        → 113952 lignes, 20 colonnes

 Mapping UAVIDS-2025

  Dimension M (10 features) :
    - TxPackets
    - RxPackets
    - LostPackets
    - TxBytes
    - RxBytes
    - TxPacketRate/s
    - RxPacketRate/s
    - TxByteRate/s
    - RxByteRate/s
    - PacketDropRate

  Dimension C (3 features) :
    - Protocol
    - SrcPort
    - DstPort

  Dimension P (6 features) :
    - FlowDuration/s
    - MeanDelay/s
    - MeanJitter/s
    - Throughput/Kbps
    - MeanPacketSize
    - Avera